# Snowflake Security Checks -> Google SecOps

Run selected Snowflake SQL checks across one or more accounts, preview the SecOps delivery plan,
and send selected account/query result groups to Google SecOps `importPushLogs` as newline-delimited JSON.

## Workflow
1. Install dependencies.
2. Load `.env` configuration.
3. Choose accounts and queries in the UI.
4. Review the SecOps dry-run preview.
5. Run the selected queries.
6. Choose which account/query groups to send.
7. Send the selected groups to Google SecOps.

In [ ]:
# Cell 2 - Install dependencies
%pip install --quiet "snowflake-connector-python[pandas]>=3.16.0" "requests>=2.31.0" "python-dotenv>=1.0.0" "tenacity>=8.2.0" "pandas>=2.0.0" "ipywidgets>=8.0.0"
print("Dependencies installed.")


## Step 1 - Configuration

Configure Snowflake accounts with `SNOWFLAKE_*_<n>` variables and SQL checks with
`SECURITY_CHECK_NAME_<n>` / `SECURITY_CHECK_SQL_<n>` in `.env`.

In [ ]:
# Cell 4 - Configuration helpers
import getpass
import os
from pathlib import Path
from typing import Optional
from urllib.parse import parse_qsl, parse_qs, urlencode, urlparse, urlunparse

from dotenv import load_dotenv


NOTEBOOK_FILENAME = "snowflake_trust_center_to_secops.ipynb"
ACCOUNT_ENV_PREFIXES = (
    "SNOWFLAKE_ACCOUNT_",
    "SNOWFLAKE_USER_",
    "SNOWFLAKE_PRIVATE_KEY_PATH_",
    "SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_",
    "SNOWFLAKE_ROLE_",
    "SNOWFLAKE_WAREHOUSE_",
    "SNOWFLAKE_LABEL_",
)
SECURITY_CHECK_ENV_PREFIXES = (
    "SECURITY_CHECK_NAME_",
    "SECURITY_CHECK_SQL_",
)


def detect_project_dir() -> Path:
    """Resolve the notebook directory or fall back to the current working directory."""
    override = os.getenv("NOTEBOOK_ROOT", "").strip()
    if override:
        return Path(override).expanduser().resolve()

    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        if (root / NOTEBOOK_FILENAME).exists():
            return root.resolve()
    return Path.cwd().resolve()


def resolve_env_path(project_dir: Optional[Path] = None) -> Path:
    """Resolve the configuration file path."""
    project_dir = project_dir or detect_project_dir()
    env_path = os.getenv("CONFIG_ENV_PATH", str(project_dir / ".env"))
    return Path(env_path).expanduser().resolve()


def load_environment(env_path: Optional[Path] = None) -> tuple[Path, Path]:
    """Load `.env` into the process environment and return the resolved paths."""
    project_dir = detect_project_dir()
    resolved_env_path = Path(env_path or resolve_env_path(project_dir)).expanduser().resolve()
    load_dotenv(resolved_env_path, override=False)
    return project_dir, resolved_env_path


def normalize_private_key_path(raw_path: str) -> str:
    """Normalize the configured private key path for display and connector use."""
    raw_path = (raw_path or "").strip()
    if not raw_path:
        return ""
    return str(Path(raw_path).expanduser().resolve())


def decode_multiline_env_value(raw_value: str) -> str:
    """Turn escaped line breaks from `.env` into real newlines."""
    value = (raw_value or "").strip()
    if not value:
        return ""
    return value.replace("\\n", "\n").replace("\\t", "\t")


def collect_suffix_indexes(environ: dict, prefixes: tuple[str, ...]) -> list[int]:
    """Collect numeric suffixes for the provided environment variable prefixes."""
    indexes = set()
    for key in environ:
        for prefix in prefixes:
            if key.startswith(prefix):
                suffix = key[len(prefix) :]
                if suffix.isdigit():
                    indexes.add(int(suffix))
    return sorted(indexes)


def build_accounts(environ: dict, prompt_for_missing: bool = True) -> list[dict]:
    """Build Snowflake account configuration from environment variables."""
    indexes = collect_suffix_indexes(environ, ACCOUNT_ENV_PREFIXES)
    accounts = []
    for idx in indexes:
        account = environ.get(f"SNOWFLAKE_ACCOUNT_{idx}", "").strip()
        user = environ.get(f"SNOWFLAKE_USER_{idx}", "").strip()
        private_key_path = normalize_private_key_path(
            environ.get(f"SNOWFLAKE_PRIVATE_KEY_PATH_{idx}", "")
        )
        private_key_passphrase = environ.get(
            f"SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_{idx}", ""
        )
        role = environ.get(f"SNOWFLAKE_ROLE_{idx}", "").strip()
        warehouse = environ.get(f"SNOWFLAKE_WAREHOUSE_{idx}", "").strip()
        label = environ.get(f"SNOWFLAKE_LABEL_{idx}", "").strip() or f"account_{idx}"

        if not account:
            continue

        missing = []
        if not user:
            missing.append(f"SNOWFLAKE_USER_{idx}")
        if not private_key_path:
            missing.append(f"SNOWFLAKE_PRIVATE_KEY_PATH_{idx}")
        if missing:
            raise ValueError(
                f"Snowflake account {idx} is missing required values: {', '.join(missing)}"
            )

        accounts.append(
            {
                "index": idx,
                "key": f"account_{idx}",
                "label": label,
                "account": account,
                "user": user,
                "private_key_path": private_key_path,
                "private_key_passphrase": private_key_passphrase,
                "role": role,
                "warehouse": warehouse,
            }
        )

    if accounts:
        return accounts
    if not prompt_for_missing:
        raise ValueError("At least one Snowflake account is required.")

    account = input("Snowflake account locator: ").strip()
    user = input("Snowflake user: ").strip()
    private_key_path = normalize_private_key_path(input("Snowflake private key path: ").strip())
    private_key_passphrase = getpass.getpass(
        "Snowflake private key passphrase (leave blank if not encrypted): "
    )
    if not account or not user or not private_key_path:
        raise ValueError("Snowflake account, user, and private key path are required.")
    return [
        {
            "index": 1,
            "key": "account_1",
            "label": "account_1",
            "account": account,
            "user": user,
            "private_key_path": private_key_path,
            "private_key_passphrase": private_key_passphrase,
            "role": "",
            "warehouse": "",
        }
    ]


def build_security_checks(environ: dict) -> list[dict]:
    """Build configured SQL checks from environment variables."""
    indexes = collect_suffix_indexes(environ, SECURITY_CHECK_ENV_PREFIXES)
    checks = []
    for idx in indexes:
        raw_sql = environ.get(f"SECURITY_CHECK_SQL_{idx}", "")
        query_name = environ.get(f"SECURITY_CHECK_NAME_{idx}", "").strip() or f"security_check_{idx}"
        sql = decode_multiline_env_value(raw_sql)

        if not sql:
            if environ.get(f"SECURITY_CHECK_NAME_{idx}", "").strip():
                raise ValueError(
                    f"SECURITY_CHECK_NAME_{idx} is set but SECURITY_CHECK_SQL_{idx} is blank."
                )
            continue

        checks.append(
            {
                "index": idx,
                "key": f"security_check_{idx}",
                "name": query_name,
                "sql": sql,
            }
        )

    if not checks:
        raise ValueError("At least one SECURITY_CHECK_SQL_<n> value is required.")
    return checks


def url_has_query_credential(url: str, key: str) -> bool:
    """Return True when the webhook URL already carries the requested auth parameter."""
    values = parse_qs(urlparse(url).query).get(key, [])
    return any(value.strip() for value in values)


def describe_auth_mode(url: str) -> str:
    """Describe how the webhook is authenticated."""
    has_key = url_has_query_credential(url, "key")
    has_secret = url_has_query_credential(url, "secret")
    if has_key and has_secret:
        return "url_query"
    if (not has_key) and (not has_secret):
        return "headers"
    return "mixed"


def mask_webhook_url(url: str) -> str:
    """Mask key and secret values when rendering the webhook URL."""
    if not url:
        return "<unset>"
    parsed = urlparse(url)
    masked_items = []
    for item_key, item_value in parse_qsl(parsed.query, keep_blank_values=True):
        if item_key in {"key", "secret"} and item_value:
            masked_items.append((item_key, "***"))
        else:
            masked_items.append((item_key, item_value))
    return urlunparse(parsed._replace(query=urlencode(masked_items)))


def build_runtime_config(
    environ: dict,
    project_dir: Path,
    env_path: Path,
    prompt_for_missing: bool = True,
) -> dict:
    """Build the notebook runtime configuration."""
    accounts = build_accounts(environ, prompt_for_missing=prompt_for_missing)
    security_checks = build_security_checks(environ)

    webhook_url = environ.get("SECOPS_WEBHOOK_URL", "").strip()
    if not webhook_url and prompt_for_missing:
        webhook_url = input("Google SecOps webhook URL: ").strip()
    if not webhook_url:
        raise ValueError("SECOPS_WEBHOOK_URL is required.")

    config = {
        "PROJECT_DIR": str(project_dir),
        "ENV_PATH": str(env_path),
        "ENV_FOUND": env_path.exists(),
        "SNOWFLAKE_ACCOUNTS": accounts,
        "SECURITY_CHECKS": security_checks,
        "SECOPS_WEBHOOK_URL": webhook_url,
        "SECOPS_API_KEY": environ.get("SECOPS_API_KEY", "").strip(),
        "SECOPS_WEBHOOK_SECRET": environ.get("SECOPS_WEBHOOK_SECRET", "").strip(),
        "BATCH_SIZE": int(environ.get("BATCH_SIZE", "100")),
        "DRY_RUN": environ.get("DRY_RUN", "false").strip().lower() == "true",
        "QUERY_RESULTS": {},
        "SEND_SELECTION": [],
    }

    if not config["DRY_RUN"] and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "key"):
        if prompt_for_missing and not config["SECOPS_API_KEY"]:
            config["SECOPS_API_KEY"] = getpass.getpass("Google SecOps API key: ")
    if not config["DRY_RUN"] and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "secret"):
        if prompt_for_missing and not config["SECOPS_WEBHOOK_SECRET"]:
            config["SECOPS_WEBHOOK_SECRET"] = getpass.getpass(
                "Google SecOps webhook secret: "
            )

    missing_auth = []
    if (
        not config["DRY_RUN"]
        and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "key")
        and not config["SECOPS_API_KEY"]
    ):
        missing_auth.append("SECOPS_API_KEY")
    if (
        not config["DRY_RUN"]
        and not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "secret")
        and not config["SECOPS_WEBHOOK_SECRET"]
    ):
        missing_auth.append("SECOPS_WEBHOOK_SECRET")
    if missing_auth:
        raise ValueError(
            "Missing Google SecOps auth values: " + ", ".join(missing_auth)
        )

    return config


def summarize_runtime_config(config: dict) -> dict:
    """Build a concise configuration summary for notebook output."""
    return {
        "project_dir": config["PROJECT_DIR"],
        "env_path": config["ENV_PATH"],
        "env_found": config["ENV_FOUND"],
        "accounts": [
            {
                "label": account["label"],
                "account": account["account"],
                "user": account["user"],
                "role": account["role"] or "<default>",
                "warehouse": account["warehouse"] or "<default>",
            }
            for account in config["SNOWFLAKE_ACCOUNTS"]
        ],
        "security_checks": [
            {
                "key": check["key"],
                "name": check["name"],
                "sql_lines": len(check["sql"].splitlines()) or 1,
            }
            for check in config["SECURITY_CHECKS"]
        ],
        "batch_size": config["BATCH_SIZE"],
        "dry_run": config["DRY_RUN"],
        "secops_auth_mode": describe_auth_mode(config["SECOPS_WEBHOOK_URL"]),
        "secops_webhook_url": mask_webhook_url(config["SECOPS_WEBHOOK_URL"]),
    }


In [ ]:
# Cell 5 - Load configuration
import json

PROJECT_DIR, ENV_PATH = load_environment()
CONFIG = build_runtime_config(
    os.environ,
    project_dir=PROJECT_DIR,
    env_path=ENV_PATH,
    prompt_for_missing=True,
)
CONFIG_SUMMARY = summarize_runtime_config(CONFIG)
print("Configuration loaded.")
print(json.dumps(CONFIG_SUMMARY, indent=2))


## Step 2 - Account and Query Selection

Select the accounts and queries to run. The default selection keeps only the first configured account
and preselects every configured query.

In [ ]:
# Cell 7 - Selector helpers
def build_default_selection(config: dict) -> dict:
    """Build the default selector state."""
    default_accounts = []
    if config["SNOWFLAKE_ACCOUNTS"]:
        default_accounts = [config["SNOWFLAKE_ACCOUNTS"][0]["label"]]
    return {
        "accounts": default_accounts,
        "queries": [check["key"] for check in config["SECURITY_CHECKS"]],
    }


def validate_selection(config: dict, account_labels: list[str], query_keys: list[str]) -> None:
    """Validate selector values against the configured options."""
    available_accounts = {account["label"] for account in config["SNOWFLAKE_ACCOUNTS"]}
    available_queries = {check["key"] for check in config["SECURITY_CHECKS"]}

    if not account_labels:
        raise ValueError("Select at least one Snowflake account.")
    if not query_keys:
        raise ValueError("Select at least one security check.")

    invalid_accounts = sorted(set(account_labels) - available_accounts)
    invalid_queries = sorted(set(query_keys) - available_queries)
    if invalid_accounts:
        raise ValueError(f"Unknown Snowflake account labels: {invalid_accounts}")
    if invalid_queries:
        raise ValueError(f"Unknown security check keys: {invalid_queries}")


def get_selection_state(account_widget, query_widget, config: dict) -> dict:
    """Read and validate the current widget selection."""
    account_labels = list(account_widget.value)
    query_keys = list(query_widget.value)
    validate_selection(config, account_labels, query_keys)
    return {"accounts": account_labels, "queries": query_keys}


def build_selection_summary(config: dict, selection: dict) -> dict:
    """Build a summary of the currently selected accounts and queries."""
    query_lookup = {check["key"]: check for check in config["SECURITY_CHECKS"]}
    return {
        "selected_accounts": list(selection["accounts"]),
        "selected_queries": [
            {"key": query_key, "name": query_lookup[query_key]["name"]}
            for query_key in selection["queries"]
        ],
        "selected_group_count": len(selection["accounts"]) * len(selection["queries"]),
    }


def build_secops_preview(config: dict, selection: dict) -> dict:
    """Build the SecOps dry-run preview payload."""
    summary = build_selection_summary(config, selection)
    summary.update(
        {
            "auth_mode": describe_auth_mode(config["SECOPS_WEBHOOK_URL"]),
            "masked_webhook_url": mask_webhook_url(config["SECOPS_WEBHOOK_URL"]),
            "payload_format": "NDJSON generic JSON logs via importPushLogs",
        }
    )
    return summary


In [ ]:
# Cell 8 - Account and query selector UI
import ipywidgets as widgets
from IPython.display import clear_output, display

DEFAULT_SELECTION = build_default_selection(CONFIG)
CURRENT_SELECTION = dict(DEFAULT_SELECTION)

account_options = [
    (f"{account['label']} ({account['account']})", account["label"])
    for account in CONFIG["SNOWFLAKE_ACCOUNTS"]
]
query_options = [
    (f"{check['name']} [{check['key']}]", check["key"])
    for check in CONFIG["SECURITY_CHECKS"]
]

ACCOUNT_SELECTOR = widgets.SelectMultiple(
    options=account_options,
    value=tuple(DEFAULT_SELECTION["accounts"]),
    description="Accounts",
    rows=max(4, min(len(account_options), 8)),
    layout=widgets.Layout(width="100%"),
)
QUERY_SELECTOR = widgets.SelectMultiple(
    options=query_options,
    value=tuple(DEFAULT_SELECTION["queries"]),
    description="Queries",
    rows=max(6, min(len(query_options), 12)),
    layout=widgets.Layout(width="100%"),
)
SELECTION_OUTPUT = widgets.Output()


def refresh_selection_output(*_args) -> None:
    """Render the current selector state."""
    global CURRENT_SELECTION
    with SELECTION_OUTPUT:
        clear_output(wait=True)
        CURRENT_SELECTION = get_selection_state(ACCOUNT_SELECTOR, QUERY_SELECTOR, CONFIG)
        summary = build_selection_summary(CONFIG, CURRENT_SELECTION)
        print("Current selection:")
        print(f"  Accounts : {summary['selected_accounts']}")
        print(
            "  Queries  : "
            + str([item['name'] for item in summary['selected_queries']])
        )
        print(f"  Groups   : {summary['selected_group_count']}")


ACCOUNT_SELECTOR.observe(refresh_selection_output, names="value")
QUERY_SELECTOR.observe(refresh_selection_output, names="value")

display(
    widgets.VBox(
        [
            widgets.HTML("<b>Snowflake accounts</b>"),
            ACCOUNT_SELECTOR,
            widgets.HTML("<b>Security checks</b>"),
            QUERY_SELECTOR,
            SELECTION_OUTPUT,
        ]
    )
)
refresh_selection_output()


In [ ]:
# Cell 9 - Google SecOps dry-run preview
import json

SECOPS_PREVIEW = build_secops_preview(CONFIG, CURRENT_SELECTION)
print("Google SecOps preview:")
print(json.dumps(SECOPS_PREVIEW, indent=2))
print("No network request is sent in this preview step.")


## Step 3 - Snowflake Connectivity Check

Run a lightweight session check for the currently selected accounts before executing the configured queries.

In [ ]:
# Cell 11 - Snowflake helpers
from pathlib import Path

import pandas as pd
import snowflake.connector


def connect_snowflake(acct_cfg: dict):
    """Create a Snowflake connection using key pair authentication."""
    private_key_path = Path(acct_cfg["private_key_path"]).expanduser().resolve()
    if not private_key_path.exists():
        raise FileNotFoundError(
            f"[{acct_cfg['label']}] Private key file not found: {private_key_path}"
        )

    connect_kwargs = {
        "account": acct_cfg["account"],
        "user": acct_cfg["user"],
        "authenticator": "SNOWFLAKE_JWT",
        "private_key_file": str(private_key_path),
        "private_key_file_pwd": acct_cfg.get("private_key_passphrase") or None,
        "client_session_keep_alive": False,
        "network_timeout": 30,
    }
    if acct_cfg.get("role"):
        connect_kwargs["role"] = acct_cfg["role"]
    if acct_cfg.get("warehouse"):
        connect_kwargs["warehouse"] = acct_cfg["warehouse"]
    return snowflake.connector.connect(**connect_kwargs)


def close_connection(conn) -> None:
    """Close a Snowflake connection when possible."""
    if conn is None:
        return
    close = getattr(conn, "close", None)
    if callable(close):
        close()


def fetch_query_dataframe(conn, sql: str) -> pd.DataFrame:
    """Execute one SQL statement and return the result as a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql)
        if not cur.description:
            return pd.DataFrame()
        columns = [column[0] for column in cur.description]
        frames = []
        try:
            for batch in cur.fetch_pandas_batches():
                frames.append(batch.reindex(columns=columns))
        except Exception:
            rows = cur.fetchall()
            frames = [pd.DataFrame(rows, columns=columns)]
    if not frames:
        return pd.DataFrame(columns=columns)
    return pd.concat(frames, ignore_index=True).reindex(columns=columns)


def run_selected_account_sanity_checks(
    config: dict,
    account_labels: list[str],
    connect_fn=None,
    close_fn=None,
) -> list[dict]:
    """Run a basic session query for the selected accounts."""
    if not config["SECURITY_CHECKS"]:
        raise ValueError("At least one configured security check is required.")
    validate_selection(config, account_labels, [config["SECURITY_CHECKS"][0]["key"]])
    account_lookup = {account["label"]: account for account in config["SNOWFLAKE_ACCOUNTS"]}
    connect_fn = connect_fn or connect_snowflake
    close_fn = close_fn or close_connection
    results = []
    for account_label in account_labels:
        conn = None
        try:
            conn = connect_fn(account_lookup[account_label])
            with conn.cursor() as cur:
                cur.execute(
                    "SELECT CURRENT_ACCOUNT(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_VERSION()"
                )
                current_account, current_role, current_warehouse, current_version = cur.fetchone()
            results.append(
                {
                    "account_label": account_label,
                    "status": "success",
                    "current_account": current_account,
                    "current_role": current_role,
                    "current_warehouse": current_warehouse,
                    "current_version": current_version,
                }
            )
        except Exception as exc:
            results.append(
                {
                    "account_label": account_label,
                    "status": "error",
                    "error": str(exc),
                }
            )
        finally:
            close_fn(conn)
    return results


In [ ]:
# Cell 12 - Selected account sanity check
CURRENT_SELECTION = get_selection_state(ACCOUNT_SELECTOR, QUERY_SELECTOR, CONFIG)
SANITY_CHECK_RESULTS = run_selected_account_sanity_checks(
    CONFIG,
    CURRENT_SELECTION["accounts"],
)
print("Snowflake connectivity check:")
for result in SANITY_CHECK_RESULTS:
    if result["status"] == "success":
        print(
            f"  [OK] {result['account_label']}: account={result['current_account']}, "
            f"role={result['current_role']}, warehouse={result['current_warehouse']}, "
            f"version={result['current_version']}"
        )
    else:
        print(f"  [ERROR] {result['account_label']}: {result['error']}")


## Step 4 - Execute the Selected Queries

Each selected account/query pair runs independently. Failures and empty result sets are recorded per group.

In [ ]:
# Cell 14 - Query execution helpers
import pandas as pd


def make_result_group_key(account_label: str, query_key: str) -> str:
    """Build a stable key for one account/query result group."""
    return f"{account_label}::{query_key}"


def execute_selected_queries(
    config: dict,
    selection: dict,
    connect_fn=None,
    fetch_fn=None,
    close_fn=None,
) -> dict:
    """Execute the selected account/query combinations."""
    validate_selection(config, selection["accounts"], selection["queries"])
    account_lookup = {account["label"]: account for account in config["SNOWFLAKE_ACCOUNTS"]}
    query_lookup = {check["key"]: check for check in config["SECURITY_CHECKS"]}
    connect_fn = connect_fn or connect_snowflake
    fetch_fn = fetch_fn or fetch_query_dataframe
    close_fn = close_fn or close_connection

    results = {}
    for account_label in selection["accounts"]:
        for query_key in selection["queries"]:
            query_def = query_lookup[query_key]
            group_key = make_result_group_key(account_label, query_key)
            conn = None
            result = {
                "group_key": group_key,
                "account_label": account_label,
                "query_key": query_key,
                "query_name": query_def["name"],
                "status": "pending",
                "row_count": 0,
                "columns": [],
                "dataframe": pd.DataFrame(),
                "error": None,
            }
            try:
                conn = connect_fn(account_lookup[account_label])
                dataframe = fetch_fn(conn, query_def["sql"])
                dataframe = dataframe.copy()
                result["dataframe"] = dataframe
                result["columns"] = dataframe.columns.tolist()
                result["row_count"] = len(dataframe)
                result["status"] = "empty" if dataframe.empty else "success"
            except Exception as exc:
                result["status"] = "error"
                result["error"] = str(exc)
            finally:
                close_fn(conn)
            results[group_key] = result
    return results


def summarize_query_results(query_results: dict) -> dict:
    """Summarize query execution outcomes."""
    summary = {
        "groups": len(query_results),
        "success": 0,
        "empty": 0,
        "error": 0,
        "rows": 0,
    }
    for result in query_results.values():
        status = result["status"]
        if status in summary:
            summary[status] += 1
        summary["rows"] += result.get("row_count", 0)
    return summary


In [ ]:
# Cell 15 - Execute selected queries
CURRENT_SELECTION = get_selection_state(ACCOUNT_SELECTOR, QUERY_SELECTOR, CONFIG)
EXECUTION_SELECTION = dict(CURRENT_SELECTION)
QUERY_RESULTS = execute_selected_queries(CONFIG, EXECUTION_SELECTION)
CONFIG["QUERY_RESULTS"] = QUERY_RESULTS
QUERY_RESULT_SUMMARY = summarize_query_results(QUERY_RESULTS)
print("Query execution summary:")
print(QUERY_RESULT_SUMMARY)

for group_key, result in QUERY_RESULTS.items():
    print(
        f"\n[{group_key}] status={result['status']} row_count={result['row_count']}"
        + (f" error={result['error']}" if result['error'] else "")
    )
    if result["status"] == "success":
        display(result["dataframe"].head())


## Step 5 - Select the Result Groups to Send

Successful non-empty groups are preselected. Empty and failed groups stay visible but cannot be sent.

In [ ]:
# Cell 17 - Send selection helpers
def build_send_candidates(query_results: dict) -> list[dict]:
    """Build UI metadata for the send selection step."""
    candidates = []
    for group_key in sorted(query_results):
        result = query_results[group_key]
        selectable = result["status"] == "success" and result["row_count"] > 0
        if result["status"] == "error":
            reason = f"error: {result['error']}"
        else:
            reason = f"{result['row_count']} row(s)"
        candidates.append(
            {
                "group_key": group_key,
                "account_label": result["account_label"],
                "query_key": result["query_key"],
                "query_name": result["query_name"],
                "row_count": result["row_count"],
                "disabled": not selectable,
                "default_selected": selectable,
                "reason": reason,
                "label": (
                    f"[{result['account_label']}] {result['query_name']} "
                    f"[{result['query_key']}]"
                ),
            }
        )
    return candidates


def get_send_selection(send_checkboxes: dict, candidates: list[dict]) -> list[str]:
    """Read the currently selected account/query result groups."""
    candidate_lookup = {candidate["group_key"]: candidate for candidate in candidates}
    selected = []
    for group_key, checkbox in send_checkboxes.items():
        candidate = candidate_lookup[group_key]
        if candidate["disabled"]:
            continue
        if checkbox.value:
            selected.append(group_key)
    return selected


In [ ]:
# Cell 18 - Send selection UI
import ipywidgets as widgets
from IPython.display import clear_output, display

if not CONFIG.get("QUERY_RESULTS"):
    raise RuntimeError("Run the query execution cell before building the send selection UI.")

SEND_CANDIDATES = build_send_candidates(CONFIG["QUERY_RESULTS"])
SEND_CHECKBOXES = {}
SEND_SELECTION_OUTPUT = widgets.Output()
rows = []
for candidate in SEND_CANDIDATES:
    checkbox = widgets.Checkbox(
        value=candidate["default_selected"],
        description=candidate["label"],
        disabled=candidate["disabled"],
        indent=False,
        layout=widgets.Layout(width="70%"),
    )
    SEND_CHECKBOXES[candidate["group_key"]] = checkbox
    reason = widgets.HTML(
        value=f"<span style='color:#666'>{candidate['reason']}</span>",
        layout=widgets.Layout(width="30%"),
    )
    rows.append(widgets.HBox([checkbox, reason]))


def refresh_send_selection_output(*_args) -> None:
    """Render the currently selected result groups."""
    with SEND_SELECTION_OUTPUT:
        clear_output(wait=True)
        selected_groups = get_send_selection(SEND_CHECKBOXES, SEND_CANDIDATES)
        CONFIG["SEND_SELECTION"] = selected_groups
        print("Selected groups to send:")
        print(selected_groups)


for checkbox in SEND_CHECKBOXES.values():
    checkbox.observe(refresh_send_selection_output, names="value")

display(widgets.VBox(rows + [SEND_SELECTION_OUTPUT]))
refresh_send_selection_output()


## Step 6 - Build Payloads and Send to Google SecOps

Each query result row becomes one top-level JSON object. The sender uses newline-delimited JSON and keeps
the existing batching and retry behavior.

In [ ]:
# Cell 20 - SecOps sender helpers
import json
import logging
from datetime import date, datetime, timezone
from decimal import Decimal
from email.utils import parsedate_to_datetime
from typing import Optional

import pandas as pd
import requests
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)


TIMESTAMP_FIELD_CANDIDATES = (
    "event_timestamp",
    "timestamp",
    "occurred_at",
    "completed_at",
    "created_at",
    "updated_at",
)
logging.basicConfig(level=logging.WARNING)
_log = logging.getLogger("secops_sender")
_DEFAULT_SECOPS_WAIT = wait_exponential(multiplier=1, min=2, max=30)
_MAX_REQUEST_BYTES = (4 * 1024 * 1024) - 1024


class RetryableSecOpsError(Exception):
    """Raised for retryable webhook responses."""

    def __init__(self, message: str, retry_after: Optional[int] = None):
        super().__init__(message)
        self.retry_after = retry_after


class NonRetryableSecOpsError(Exception):
    """Raised for permanent webhook failures."""


def json_safe_value(value):
    """Convert notebook values into JSON-safe primitives."""
    if isinstance(value, pd.Timestamp):
        if pd.isna(value):
            return None
        if value.tzinfo is None:
            value = value.tz_localize(timezone.utc)
        return value.tz_convert(timezone.utc).isoformat()
    if isinstance(value, datetime):
        if value.tzinfo is None:
            value = value.replace(tzinfo=timezone.utc)
        return value.astimezone(timezone.utc).isoformat()
    if isinstance(value, date):
        return value.isoformat()
    if isinstance(value, Decimal):
        if value == value.to_integral():
            return int(value)
        return float(value)
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, dict):
        return {str(key): json_safe_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe_value(item) for item in value]
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    item = getattr(value, "item", None)
    if callable(item):
        try:
            unpacked = item()
        except Exception:
            unpacked = value
        if unpacked is not value:
            return json_safe_value(unpacked)
    return value


def extract_source_timestamp(record: dict) -> Optional[str]:
    """Extract a source timestamp from a result record when one is available."""
    lower_map = {str(key).lower(): key for key in record}
    for candidate in TIMESTAMP_FIELD_CANDIDATES:
        original_key = lower_map.get(candidate)
        if original_key is None:
            continue
        parsed = pd.to_datetime(record.get(original_key), utc=True, errors="coerce")
        if pd.isna(parsed):
            continue
        return parsed.isoformat()
    return None


def row_to_generic_log(
    record: dict,
    account_label: str,
    query_key: str,
    query_name: str,
    row_index: int,
    generated_at: Optional[str] = None,
) -> dict:
    """Convert one query result row into a generic SecOps log object."""
    generated_at = generated_at or datetime.now(timezone.utc).isoformat()
    safe_record = {str(key): json_safe_value(value) for key, value in record.items()}
    event = {
        "snowflake_account": account_label,
        "snowflake_query_key": query_key,
        "snowflake_query_name": query_name,
        "generated_at": generated_at,
        "row_index": row_index,
        "result": safe_record,
    }
    source_timestamp = extract_source_timestamp(record)
    if source_timestamp:
        event["source_timestamp"] = source_timestamp
    return event


def build_group_events(query_result: dict, generated_at: Optional[str] = None) -> list[dict]:
    """Convert a query result group into generic SecOps log objects."""
    dataframe = query_result["dataframe"]
    if dataframe.empty:
        return []
    generated_at = generated_at or datetime.now(timezone.utc).isoformat()
    events = []
    for row_index, record in enumerate(dataframe.to_dict(orient="records"), start=1):
        events.append(
            row_to_generic_log(
                record,
                account_label=query_result["account_label"],
                query_key=query_result["query_key"],
                query_name=query_result["query_name"],
                row_index=row_index,
                generated_at=generated_at,
            )
        )
    return events


def build_selected_events(query_results: dict, selected_group_keys: list[str]) -> tuple[list[dict], dict]:
    """Build payload events for the selected account/query groups."""
    generated_at = datetime.now(timezone.utc).isoformat()
    events = []
    group_summaries = []
    for group_key in selected_group_keys:
        result = query_results.get(group_key)
        if not result:
            continue
        if result["status"] != "success" or result["row_count"] == 0:
            continue
        group_events = build_group_events(result, generated_at=generated_at)
        events.extend(group_events)
        group_summaries.append(
            {
                "group_key": group_key,
                "account_label": result["account_label"],
                "query_key": result["query_key"],
                "query_name": result["query_name"],
                "row_count": len(group_events),
            }
        )
    return events, {
        "selected_groups": group_summaries,
        "total_events": len(events),
        "generated_at": generated_at,
    }


def _parse_retry_after(value: Optional[str]) -> Optional[int]:
    """Parse the HTTP Retry-After header."""
    if not value:
        return None
    try:
        return max(0, int(value))
    except ValueError:
        pass
    try:
        retry_at = parsedate_to_datetime(value)
        if retry_at.tzinfo is None:
            retry_at = retry_at.replace(tzinfo=timezone.utc)
        return max(0, int((retry_at - datetime.now(timezone.utc)).total_seconds()))
    except Exception:
        return None


def _secops_wait(retry_state) -> float:
    """Use Retry-After when it is available."""
    exc = retry_state.outcome.exception() if retry_state.outcome else None
    if isinstance(exc, RetryableSecOpsError) and exc.retry_after is not None:
        return exc.retry_after
    return _DEFAULT_SECOPS_WAIT(retry_state)


def _build_headers(config: dict) -> dict:
    """Build SecOps webhook headers."""
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "snowflake-security-checks-secops/1.0",
    }
    if not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "key"):
        if not config.get("SECOPS_API_KEY"):
            raise NonRetryableSecOpsError(
                "SECOPS_API_KEY is required because the webhook URL does not include key=."
            )
        headers["X-goog-api-key"] = config["SECOPS_API_KEY"]
    if not url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "secret"):
        if not config.get("SECOPS_WEBHOOK_SECRET"):
            raise NonRetryableSecOpsError(
                "SECOPS_WEBHOOK_SECRET is required because the webhook URL does not include secret=."
            )
        headers["X-Webhook-Access-Key"] = config["SECOPS_WEBHOOK_SECRET"]
    return headers


def _serialize_event(event: dict) -> bytes:
    """Serialize one event as JSON bytes."""
    return json.dumps(event, separators=(",", ":"), ensure_ascii=False).encode("utf-8")


def _build_event_batches(events: list[dict], batch_size: int) -> list[dict]:
    """Split events by count and payload size."""
    if batch_size < 1:
        raise ValueError("BATCH_SIZE must be >= 1.")

    batches = []
    current_lines = []
    current_bytes = 0
    for event_index, event in enumerate(events, start=1):
        line = _serialize_event(event)
        if len(line) > _MAX_REQUEST_BYTES:
            raise NonRetryableSecOpsError(
                f"Event {event_index} is {len(line)} bytes, exceeding the 4 MB webhook limit."
            )
        line_bytes = len(line) + (1 if current_lines else 0)
        if current_lines and (
            len(current_lines) >= batch_size or current_bytes + line_bytes > _MAX_REQUEST_BYTES
        ):
            payload = b"\n".join(current_lines)
            batches.append(
                {
                    "events": len(current_lines),
                    "bytes": len(payload),
                    "payload": payload,
                }
            )
            current_lines = [line]
            current_bytes = len(line)
        else:
            current_lines.append(line)
            current_bytes += line_bytes

    if current_lines:
        payload = b"\n".join(current_lines)
        batches.append(
            {
                "events": len(current_lines),
                "bytes": len(payload),
                "payload": payload,
            }
        )
    return batches


@retry(
    stop=stop_after_attempt(4),
    wait=_secops_wait,
    retry=retry_if_exception_type((requests.RequestException, RetryableSecOpsError)),
    before_sleep=before_sleep_log(_log, logging.WARNING),
    reraise=True,
)
def _post_batch(url: str, headers: dict, payload: bytes, post_fn=requests.post):
    """POST one NDJSON batch to Google SecOps."""
    response = post_fn(url, headers=headers, data=payload, timeout=60)
    if response.status_code in (408, 429) or response.status_code >= 500:
        retry_after = _parse_retry_after(response.headers.get("Retry-After"))
        raise RetryableSecOpsError(
            f"Retryable error {response.status_code}: {response.text[:300]}",
            retry_after=retry_after,
        )
    if response.status_code >= 400:
        raise NonRetryableSecOpsError(
            f"Client error {response.status_code}: {response.text[:500]}\n"
            "  -> 401/403: check webhook URL query auth or fallback headers\n"
            "  -> 413: reduce BATCH_SIZE or trim oversized events"
        )
    return response


def send_to_secops(events: list[dict], config: dict, post_fn=requests.post) -> dict:
    """Send the prepared events to Google SecOps or render a dry-run preview."""
    total = len(events)
    auth_mode = describe_auth_mode(config["SECOPS_WEBHOOK_URL"])
    if total == 0:
        print("No events selected for Google SecOps delivery.")
        return {
            "sent": 0,
            "batches": 0,
            "attempted_batches": 0,
            "failed_batches": 0,
            "failed_event_count": 0,
            "planned_batches": 0,
            "planned_bytes": 0,
            "auth_mode": auth_mode,
            "dry_run": config["DRY_RUN"],
            "skipped": True,
            "errors": [],
        }

    batches = _build_event_batches(events, config["BATCH_SIZE"])
    planned_bytes = sum(batch["bytes"] for batch in batches)

    if config["DRY_RUN"]:
        first_batch = batches[0]
        preview = first_batch["payload"][:1500].decode("utf-8", errors="replace")
        print(f"[DRY_RUN] {total} event(s) would be sent to {mask_webhook_url(config['SECOPS_WEBHOOK_URL'])}")
        print(f"Auth mode   : {auth_mode}")
        print(f"Batches     : {len(batches)}")
        print(f"Total bytes : {planned_bytes}")
        print(
            f"First batch : {first_batch['events']} event(s) / {first_batch['bytes']} bytes"
        )
        print("\nFirst batch preview:")
        print(preview)
        if len(first_batch["payload"]) > 1500:
            print("... <truncated>")
        return {
            "sent": 0,
            "batches": 0,
            "attempted_batches": 0,
            "failed_batches": 0,
            "failed_event_count": 0,
            "planned_batches": len(batches),
            "planned_bytes": planned_bytes,
            "auth_mode": auth_mode,
            "dry_run": True,
            "skipped": False,
            "errors": [],
        }

    headers = _build_headers(config)
    sent = 0
    successful_batches = 0
    failed_batches = 0
    failed_event_count = 0
    errors = []
    attempted_batches = 0

    for batch_index, batch in enumerate(batches, start=1):
        attempted_batches = batch_index
        try:
            _post_batch(config["SECOPS_WEBHOOK_URL"], headers, batch["payload"], post_fn=post_fn)
            sent += batch["events"]
            successful_batches += 1
            print(
                f"  Batch {batch_index}: sent {batch['events']} event(s) / {batch['bytes']} bytes "
                f"(total {sent}/{total})"
            )
        except Exception as exc:
            failed_batches += 1
            failed_event_count += batch["events"]
            errors.append(
                {
                    "batch": batch_index,
                    "events": batch["events"],
                    "bytes": batch["bytes"],
                    "error": str(exc),
                }
            )
            print(f"  Batch {batch_index}: failed after retries - {exc}")
            break

    if failed_batches:
        print(
            f"Stopped after {sent} sent event(s); {failed_event_count} event(s) remain unsent."
        )
    else:
        print(f"Complete: {sent} event(s) sent in {successful_batches} batch(es).")

    return {
        "sent": sent,
        "batches": successful_batches,
        "attempted_batches": attempted_batches,
        "failed_batches": failed_batches,
        "failed_event_count": failed_event_count,
        "planned_batches": len(batches),
        "planned_bytes": planned_bytes,
        "auth_mode": auth_mode,
        "dry_run": False,
        "skipped": False,
        "errors": errors,
    }


In [ ]:
# Cell 21 - Build payloads and send selected groups
import json

if "SEND_CHECKBOXES" not in globals():
    raise RuntimeError("Run the send selection UI cell before sending data to Google SecOps.")

CONFIG["SEND_SELECTION"] = get_send_selection(SEND_CHECKBOXES, SEND_CANDIDATES)
SELECTED_EVENTS, PAYLOAD_PLAN = build_selected_events(
    CONFIG["QUERY_RESULTS"],
    CONFIG["SEND_SELECTION"],
)
print("Payload plan:")
print(json.dumps(PAYLOAD_PLAN, indent=2))

SECOPS_RESULT = send_to_secops(SELECTED_EVENTS, CONFIG)
print("\nSecOps delivery result:")
print(json.dumps(SECOPS_RESULT, indent=2))


## Step 7 - Summary

Review the execution summary, selected result groups, and Google SecOps delivery outcome.

In [ ]:
# Cell 23 - Summary
import json

query_result_rows = []
for group_key, result in CONFIG.get("QUERY_RESULTS", {}).items():
    query_result_rows.append(
        {
            "group_key": group_key,
            "account_label": result["account_label"],
            "query_key": result["query_key"],
            "query_name": result["query_name"],
            "status": result["status"],
            "row_count": result["row_count"],
            "error": result["error"],
        }
    )

final_summary = {
    "configured_accounts": [account["label"] for account in CONFIG["SNOWFLAKE_ACCOUNTS"]],
    "configured_queries": [
        {"key": check["key"], "name": check["name"]}
        for check in CONFIG["SECURITY_CHECKS"]
    ],
    "execution_selection": globals().get("EXECUTION_SELECTION", {}),
    "query_results": query_result_rows,
    "send_selection": CONFIG.get("SEND_SELECTION", []),
    "payload_plan": globals().get("PAYLOAD_PLAN", {}),
    "secops_preview": globals().get("SECOPS_PREVIEW", {}),
    "secops_result": globals().get("SECOPS_RESULT", None),
}
print(json.dumps(final_summary, indent=2))
